# 04 - Silver Clientes - Bike Store Medallion
Notebook de tratamento da camada Silver focado em clientes.

Fonte utilizada da camada Bronze:
- `bike_store.bronze.customers`

## Transformações aplicadas
- Filtro: apenas clientes com e-mail e telefone cadastrados

In [0]:
# Criando dataframe de Clientes
df_customers = spark.read.table("bike_store.bronze.customers")
df_customers.printSchema()

In [0]:
display(df_customers.head(10))

In [0]:
# Filtrando apenas clientes com registros completos
from pyspark.sql.functions import col, trim
from functools import reduce

# Aplica trim em todas as colunas string
df_customers_clean = reduce(
    lambda df, c: df.withColumn(c, trim(col(c))),
    [c for c, t in df_customers.dtypes if t == "string"],
    df_customers
)

df_silver_customers = df_customers_clean \
    .replace("NULL", None) \
    .filter(col("email").isNotNull() & col("phone").isNotNull())

print(f"Total original:  {df_customers.count()} clientes")
print(f"Após filtro:     {df_silver_customers.count()} clientes")


display(df_silver_customers.head(10))

In [0]:
# Eliminando colunas desnecessárias
df_silver_customers_clean = df_silver_customers.drop("zip_code","state")
display(df_silver_customers_clean.head(10))

In [0]:
# Salvando tabela de Clientes na camada Silver
df_silver_customers_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("bike_store.silver.clients")

print("✅ bike_store.silver.clients salvo com sucesso!")